# Gemma 3 270M: 60 seconds on 16 TPU v5e chips
Run these cells in order from the extracted pilot folder. This notebook controls an **existing Google Cloud v5e-16 training slice** from an authenticated workstation or Cloud Shell notebook. It launches every worker; the notebook itself does not need an attached TPU.

**Baseline:** random weights, native model and tokenizer, 16 sequences/update, 1,024 positions/sequence, AdamW, BF16 compute with FP32 master weights/state. Setup, compilation, evaluation, and checkpoint operations have separate timers.

Have your project ID, zone, TPU name, existing writable Cloud Storage bucket prefix, total slice hourly rate, and Hugging Face token ready. Accept the [Gemma repository terms](https://huggingface.co/google/gemma-3-270m) first. Running the launch cell uses the existing allocation and writes to your bucket. It does not create or delete a TPU.


## 1. Check the controller and create an isolated Python environment

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from getpass import getpass

ROOT = Path.cwd().resolve()
if not (ROOT / "launch_pilot.py").is_file():
    raise RuntimeError("Open this notebook from the extracted gemma-270m-tpu-pilot folder.")
if shutil.which("gcloud") is None:
    raise RuntimeError("This controller needs the authenticated Google Cloud CLI. Use a Cloud Shell notebook or an authenticated workstation.")
subprocess.run(["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=value(account)"], check=True)
TOOLING = ROOT / ".bootstrap"
if not (TOOLING / "uv").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(TOOLING), "uv==0.11.15"], check=True)
uv_env = dict(os.environ, PYTHONPATH=str(TOOLING))
uv = [sys.executable, "-m", "uv"]
PYTHON = ROOT / ".controller-venv" / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([*uv, "venv", "--python", "3.12", "--managed-python", str(PYTHON.parent.parent)], env=uv_env, check=True)
subprocess.run([*uv, "pip", "install", "--python", str(PYTHON), "-r", str(ROOT / "requirements-controller.lock")], env=uv_env, check=True)
print("Controller environment ready.")


## 2. Enter the real cloud resource details
The hourly rate must cover **all 16 chips**. The Hugging Face token is hidden and is passed only to data preparation.

In [ ]:
project = input("Google Cloud project ID: ").strip()
zone = input("Existing TPU zone: ").strip()
tpu_name = input("Existing v5e-16 TPU name: ").strip()
output_uri = input("Existing bucket/prefix for results (gs://...): ").strip().rstrip("/")
hourly_usd = float(input("Total 16-chip slice hourly rate in USD: ").strip())
if not all([project, zone, tpu_name]) or not output_uri.startswith("gs://") or hourly_usd <= 0:
    raise ValueError("Supply real resource identifiers, a gs:// results prefix, and a positive total slice rate.")
resource = json.loads(subprocess.check_output([
    "gcloud", "compute", "tpus", "tpu-vm", "describe", tpu_name,
    "--project", project, "--zone", zone, "--format=json"
], text=True))
print(json.dumps({key: resource.get(key) for key in ["name", "state", "acceleratorType", "acceleratorConfig"]}, indent=2))
if resource.get("state") != "READY":
    raise RuntimeError("The existing TPU must be READY before launch.")
prepared_env = dict(os.environ)
prepared_env["HF_TOKEN"] = os.environ.get("HF_TOKEN") or getpass("Hugging Face token with Gemma access: ")
prepared_env["TOKENIZERS_PARALLELISM"] = "false"


## 3. Prepare and freeze the data
This step runs on the controller. Existing prepared data is reused only after its hashes and configuration pass verification. No Hugging Face token is included in the worker bundle.

In [ ]:
subprocess.run([
    str(PYTHON), str(ROOT / "prepare_pilot.py"),
    "--config", str(ROOT / "config.json"), "--output", str(ROOT / "prepared"), "--reuse"
], env=prepared_env, cwd=ROOT, check=True)
prepared_env.pop("HF_TOKEN", None)
manifest = json.loads((ROOT / "prepared" / "manifest.json").read_text())
print(json.dumps({
    "model_revision": manifest["model_revision"],
    "dataset_revision": manifest["dataset_revision"],
    "training_documents": len(manifest["splits"]["train"]["documents"]),
    "validation_documents": len(manifest["splits"]["validation"]["documents"]),
    "training_blocks": manifest["splits"]["train"]["blocks"]
}, indent=2))


## 4. Launch all TPU workers
This cell installs the locked worker environment, compiles the model, runs the 60-second training interval, evaluates, and verifies a persistent checkpoint. The whole cell takes longer than one minute. Worker installation/compilation and checkpoint transfer are included in the observed-job cost estimate.

In [ ]:
launch_env = dict(os.environ)
launch_env.pop("HF_TOKEN", None)
subprocess.run([
    str(PYTHON), str(ROOT / "launch_pilot.py"),
    "--project", project, "--zone", zone, "--tpu-name", tpu_name,
    "--output-uri", output_uri, "--hourly-usd", str(hourly_usd),
    "--data-dir", str(ROOT / "prepared")
], env=launch_env, cwd=ROOT, check=True)


## 5. Inspect the measured results
Engineering PASS and Learning PASS are separate. The allocation remains active when the job returns; its bill continues until you stop or delete it.

In [ ]:
from IPython.display import display, Markdown
run_dirs = sorted(path for path in (ROOT / "runs").iterdir() if (path / "launch.json").is_file())
if not run_dirs:
    raise RuntimeError("No launch report found.")
run_dir = run_dirs[-1]
launch = json.loads((run_dir / "launch.json").read_text())
print("Persistent artifacts:", launch["output_uri"])
print("Local reports:", run_dir)
if not (run_dir / "result.json").exists():
    raise RuntimeError("No completed result. Inspect launcher.log and the failure artifacts in the results prefix.")
result = json.loads((run_dir / "result.json").read_text())
verification = json.loads((run_dir / "verification.json").read_text())
rows = [
    ("Engineering", verification["engineering_status"]),
    ("Learning", verification["learning_status"]),
    ("TPU chips", result["device_count"]),
    ("Parameters", f'{result["model_parameters"]:,}'),
    ("Compilation seconds", f'{result["compilation_seconds"]:.2f}'),
    ("Actual training seconds", f'{result["training_seconds"]:.3f}'),
    ("Completed updates", result["updates"]),
    ("Scored tokens/second", f'{result["tokens_per_second"]:,.1f}'),
    ("Initial validation loss", f'{result["initial_eval"]["loss"]:.6f}'),
    ("Final validation loss", f'{result["final_eval"]["loss"]:.6f}'),
    ("Reload loss difference", f'{result["reload_loss_error"]:.8f}'),
    ("Observed-job cost estimate USD", f'{launch["observed_job_compute_estimate_usd"]:.4f}')
]
display(Markdown("| Measurement | Result |\n|---|---|\n" + "\n".join(f"| {name} | {value} |" for name, value in rows)))
print(json.dumps(verification["requirements"], indent=2))
print(launch["cost_scope"])
